<a href="https://colab.research.google.com/github/mrdbourke/pytorch-deep-learning/blob/main/extras/exercises/05_pytorch_going_modular_exercise_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05. PyTorch Going Modular Exercises

Welcome to the 05. PyTorch Going Modular exercise template notebook.

There are several questions in this notebook and it's your goal to answer them by writing Python and PyTorch code.

> **Note:** There may be more than one solution to each of the exercises, don't worry too much about the *exact* right answer. Try to write some code that works first and then improve it if you can.

## Resources and solutions

* These exercises/solutions are based on [section 05. PyTorch Going Modular](https://www.learnpytorch.io/05_pytorch_going_modular/) of the Learn PyTorch for Deep Learning course by Zero to Mastery.

**Solutions:** 

Try to complete the code below *before* looking at these.

* See a live [walkthrough of the solutions (errors and all) on YouTube](https://youtu.be/ijgFhMK3pp4).
* See an example [solutions notebook for these exercises on GitHub](https://github.com/mrdbourke/pytorch-deep-learning/blob/main/extras/solutions/05_pytorch_going_modular_exercise_solutions.ipynb).

## 1. Turn the code to get the data (from section 1. Get Data) into a Python script, such as `get_data.py`.

* When you run the script using `python get_data.py` it should check if the data already exists and skip downloading if it does.
* If the data download is successful, you should be able to access the `pizza_steak_sushi` images from the `data` directory.

In [12]:
%%writefile get_data.py

import requests
import zipfile
from pathlib import Path
import os

url = "https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip"

data_path = Path("./exercise_data")
zip_file_name = "pizza_steak_sushi.zip"
image_path = data_path / "pizza_steak_sushi"


if not image_path.is_dir():
    image_path.mkdir(parents=True,
                     exist_ok=True)

    with open(data_path/zip_file_name, 'wb') as f:
        r = requests.get(url)
        print(f"Downloading the data zip file")
        f.write(r.content)
else:
    print("data already exists.. Skipping download")

if not os.listdir(image_path):
    with zipfile.ZipFile(data_path/zip_file_name, 'r') as z_ref:
        print(f"extracting the image files")
        z_ref.extractall(path=image_path)
else:
    print('Images are extracted. Skipping extraction')



Overwriting get_data.py


In [13]:
# Example running of get_data.py
!python get_data.py

extracting the image files


## 2. Use [Python's `argparse` module](https://docs.python.org/3/library/argparse.html) to be able to send the `train.py` custom hyperparameter values for training procedures.
* Add an argument flag for using a different:
  * Training/testing directory
  * Learning rate
  * Batch size
  * Number of epochs to train for
  * Number of hidden units in the TinyVGG model
    * Keep the default values for each of the above arguments as what they already are (as in notebook 05).
* For example, you should be able to run something similar to the following line to train a TinyVGG model with a learning rate of 0.003 and a batch size of 64 for 20 epochs: `python train.py --learning_rate 0.003 batch_size 64 num_epochs 20`.
* **Note:** Since `train.py` leverages the other scripts we created in section 05, such as, `model_builder.py`, `utils.py` and `engine.py`, you'll have to make sure they're available to use too. You can find these in the [`going_modular` folder on the course GitHub](https://github.com/mrdbourke/pytorch-deep-learning/tree/main/going_modular/going_modular). 

In [ ]:
# YOUR CODE HERE

In [14]:
image_path

WindowsPath('exercise_data/pizza_steak_sushi')

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torch import nn

train_dir = image_path / "train"
test_dir = image_path / "test"

simple_transform = transforms.Compose([transforms.Resize(64,64,3),
                                       transforms.ToTensor()])
#creating the datasets
train_dataset = datasets.ImageFolder(root=train_dir,
                                     transform=simple_transform,
                                     target_transform=None)

test_dataset = datasets.ImageFolder(root=test_dir,
                                    transform=simple_transform,
                                    target_transform=None)

device = 'cuda' if torch.cuda.is_available() else 'cpu'



#creating the tinyvgg
def train(num_epochs, batch_size, hidden_units, learning_rate):
    #defining the CNN model
    class TinyVGG(nn.Module):
        def __init__(self, input_neurons, hidden_units, output_neurons):
            super().__init__()
            self.conv_1 = nn.Sequential(nn.Conv2d(in_channels=input_neurons,
                                                out_channels=hidden_units,
                                                kernel_size=3,
                                                padding=1,
                                                stride=1),
                                        nn.ReLU(),
                                        nn.Conv2d(in_channels=hidden_units,
                                                out_channels=hidden_units,
                                                kernel_size=3,
                                                padding=1,
                                                stride=1),
                                        nn.ReLU(),
                                        nn.MaxPool2d(kernel_size=2))
            self.conv_2 = nn.Sequential(nn.Conv2d(in_channels=hidden_units,
                                                out_channels=hidden_units,
                                                kernel_size=3,
                                                padding=1,
                                                stride=1),
                                        nn.ReLU(),
                                        nn.Conv2d(in_channels=hidden_units,
                                                out_channels=hidden_units,
                                                kernel_size=3,
                                                padding=1,
                                                stride=1),
                                        nn.ReLU(),
                                        nn.MaxPool2d(kernel_size=2))
            self.classifier = nn.Sequential(nn.Flatten(),
                                            nn.Linear(in_features=hidden_units*16*16,
                                                    out_features=output_neurons))
        
        def forward(self, x):
            x = self.conv_1(x)
            x = self.conv_2(x)
            x = self.classifier(x)
            return x
    
    #Initializing the CNN
    model = TinyVGG(input_neurons=3,
                    hidden_units=hidden_units,
                    output_neurons=10)
    
    #creating the dataloader
    train_dataloader = DataLoader(dataset=train_dataset,
                                  batch_size=batch_size,
                                  shuffle=True)
    
    test_dataloader  = DataLoader(dataset=test_dataset,
                                  batch_size=batch_size)

    

        

In [ ]:
# Example running of train.py
!python train.py --num_epochs 5 --batch_size 128 --hidden_units 128 --learning_rate 0.0003

## 3. Create a Python script to predict (such as `predict.py`) on a target image given a file path with a saved model.

* For example, you should be able to run the command `python predict.py some_image.jpeg` and have a trained PyTorch model predict on the image and return its prediction.
* To see example prediction code, check out the [predicting on a custom image section in notebook 04](https://www.learnpytorch.io/04_pytorch_custom_datasets/#113-putting-custom-image-prediction-together-building-a-function). 
* You may also have to write code to load in a trained model.

In [ ]:
# YOUR CODE HERE

In [ ]:
# Example running of predict.py 
!python predict.py --image data/pizza_steak_sushi/test/sushi/175783.jpg